## Exercises on Offline Reinforcement Learning and Decision Transformers

These paper-and-pencil exercises reinforce Chapter 12: the compounding of behaviour-cloning errors, extrapolation error in offline Q-learning, the mechanics of the CQL penalty (soft maximum, gradients, lower bound), the return-to-go representation of a trajectory, causal attention and what the state token can see, and return conditioning at test time. Notation: dataset $\mathcal D$, behaviour policy $\pi_\beta$, learned $Q_\theta$, CQL coefficient $\alpha$ (not a step size), return-to-go $\hat R_t$, context length $K$, embedding size $E$.

### Exercise 12.1 — Compounding errors in behaviour cloning

A cloned policy makes a mistake with probability $\eta$ at each step while it is in states that the expert visited ("on-distribution"). After its first mistake it is off-distribution and, in the worst case, every subsequent step is wrong and costs $1$. The horizon is $T$.

1. Bound the expected number of wrong steps as a function of $\eta$ and $T$ (assume at most one on-distribution mistake matters, and ignore recovery).
2. Evaluate for $\eta=0.01$ and $T=100$, and compare with $\eta T$, the cost a supervised classifier would suggest.
3. Why does DAgger (querying the expert in the learner's own states) break this quadratic growth?

**Step 1 — Expected cost.** If the first mistake happens at step $t$ (probability $(1-\eta)^{t-1}\eta$), the remaining $T-t$ steps are wrong, costing $1+(T-t)$ in total. Bounding $(1-\eta)^{t-1}\le1$:

$\displaystyle \mathbb E[\text{cost}] \le \sum_{t=1}^{T}\eta\,(T-t+1) = \eta\,\frac{T(T+1)}{2} \approx \frac{\eta T^2}{2}.$

**Step 2 — Numbers.** $\eta=0.01$, $T=100$: the bound gives $\eta T(T+1)/2 = 50.5 \approx 50$ wrong steps out of $100$ (the exact expectation, keeping the factor $(1-\eta)^{t-1}$, is about $37$), versus $\eta T=1$ expected from the per-step error rate alone. One early mistake sends the learner into states it has never seen, and the classifier's accuracy guarantee says nothing there.

**Step 3 — DAgger.** DAgger runs the *learner*, records the states it actually visits, asks the expert what to do in those states, and retrains on the union. The learner's own off-distribution states enter the training set, so its error rate is controlled *on the distribution it induces*, and the cost returns to $O(\eta T)$. This requires an expert that can be queried on new states — exactly what the offline setting does not have.

**Key concept**

Behaviour cloning is supervised learning on the expert's state distribution, but the cloned policy is evaluated on its *own* state distribution. The mismatch makes errors compound quadratically in the horizon; fixing it requires new data from the learner's states, which offline learning cannot collect.

### Exercise 12.2 — Extrapolation error, step by step

A state $s'$ has three actions. The dataset contains only transitions with $a_1$ from $s'$, whose true value under the behaviour policy is $q_{\pi_\beta}(s',a_1)=10$. The network's estimates are $Q(s',a_1)=10$ (fitted), $Q(s',a_2)=14$ and $Q(s',a_3)=7$ (never trained, arbitrary). A transition $(s,a,r=0,s')$ is in the dataset with $\gamma=0.9$.

1. Compute the Q-learning target for $(s,a)$ with $\max_{a'}Q(s',a')$, and the target that would use only the dataset action.
2. The pair $(s,a)$ is itself the next-state pair of an earlier transition $(s^-,a^-,r=0,s)$. Suppose $Q(s,a)$ is fitted to its target. What target does $(s^-,a^-)$ receive, and how does the error propagate?
3. Online, what would happen to $Q(s',a_2)$? Offline?

**Step 1 — Two targets.** With the maximum over all actions: $y = 0+0.9\cdot\max(10,14,7) = 0.9(14) = 12.6$. Using only the action supported by the data: $y = 0.9(10) = 9.0$. The target is inflated by $3.6$ by an action whose value is pure extrapolation.

**Step 2 — Propagation.** After fitting, $Q(s,a)=12.6$ instead of $9.0$. The earlier pair's target becomes $y^- = 0.9\cdot Q(s,a) = 0.9(12.6) = 11.34$ instead of $0.9(9.0)=8.1$: the error of $3.6$ propagates backwards, discounted to $3.24$, into a pair that *is* well covered by the data. Every further target-network update pushes it one step further back, and, if $a_2$-like actions are overestimated in many states, the inflations add up along the chain.

**Step 3 — Online vs. offline.** Online, the greedy policy would choose $a_2$ in $s'$ (it looks best), observe its true value, and the estimate $14$ would be corrected downward — the overestimate is self-limiting. Offline, $a_2$ is never taken in $s'$, $Q(s',a_2)$ receives no gradient from any data, stays at $14$ (or drifts further as the network changes), and keeps feeding the targets. The error is never corrected and keeps compounding.

**Key concept**

Bootstrapped targets query $Q$ at actions the data may not support; the maximum selects their overestimates; the overestimates propagate backwards through the Bellman chain. Online, exploration corrects them; offline, nothing does — this is **extrapolation error**.

### Exercise 12.3 — The CQL penalty and its gradient

In a state $s$ with three actions the network outputs $Q(s,\cdot)=(10,\,14,\,7)$ and the dataset action is $a_1$. The CQL penalty for this state is $\alpha\big[\log\sum_a e^{Q(s,a)} - Q(s,a_1)\big]$.

1. Compute $\log\sum_a e^{Q(s,a)}$ and the penalty for $\alpha=1$.
2. Compute the gradient of the penalty with respect to each $Q(s,a)$ and interpret the signs.
3. Which action's value is pushed down most, and what happens to the greedy action after a few such updates?

**Step 1 — Soft maximum.** $\ \log\big(e^{10}+e^{14}+e^{7}\big) = 14+\log\big(e^{-4}+1+e^{-7}\big) = 14+\log(1.01923) = 14.019.$ The soft maximum is just above the hard maximum $14$. Penalty: $14.019-10 = 4.019$.

**Step 2 — Gradient.** $\partial\log\sum_b e^{Q(s,b)}/\partial Q(s,a) = \text{softmax}(Q)_a$:

$\displaystyle \text{softmax}(10,14,7) = (0.0180,\ 0.9811,\ 0.0009).$

The gradient of the penalty is therefore $(0.0180-1,\ 0.9811,\ 0.0009) = (-0.982,\ +0.981,\ +0.001)$ for $(a_1,a_2,a_3)$. Gradient *descent* on the penalty **raises** $Q(s,a_1)$ (negative gradient, the dataset action) and **lowers** $Q(s,a_2)$ (positive gradient, the overestimated out-of-distribution action); $a_3$ is barely touched because the softmax gives it no weight.

**Step 3 — Effect.** The push-down concentrates on the action the greedy policy *would choose* ($a_2$), in proportion to how confident the network is about it. After a few updates $Q(s,a_2)$ falls below $Q(s,a_1)$, the greedy action becomes the dataset action, and the maximum in the targets of earlier states uses a value the data can vouch for. The penalty keeps acting on whichever out-of-distribution action rises to the top, until the greedy policy stays within the support of the data.

**Key concept**

The CQL penalty $\log\sum_a e^{Q}-Q(s,a_{\text{data}})$ has gradient $\text{softmax}(Q)-\mathbb 1_{a_{\text{data}}}$: it pushes down the actions the current greedy policy favours and pushes up the actions in the data. For in-distribution greedy actions the two cancel; for out-of-distribution ones they do not.

### Exercise 12.4 — CQL as a lower bound

Consider the CQL objective restricted to the penalty, $\min_Q\ \alpha\big[\mathbb E_{a\sim\mu}[Q(s,a)]-\mathbb E_{a\sim\pi_\beta}[Q(s,a)]\big]$, where $\mu$ is the distribution of the learned (greedy or softmax) policy and $\pi_\beta$ the behaviour distribution in state $s$.

1. Show that the penalty is zero when $\mu=\pi_\beta$, whatever $Q$ is, and describe what its gradient does to $Q(s,a)$ when $\mu$ puts more mass than $\pi_\beta$ on an action, or less.
2. With the paper's convention of a $\tfrac12$ in front of the squared TD error, the fixed point of the full CQL update satisfies $Q_\theta(s,a) = \mathcal B^\mu Q_\theta(s,a) - \alpha\big(\tfrac{\mu(a\mid s)}{\pi_\beta(a\mid s)}-1\big)$, where $\mathcal B^\mu$ is the Bellman backup of the policy $\mu$. For a state with $\pi_\beta=(0.8,0.2)$ and $\mu=(0.2,0.8)$, which action is penalised and by how much (in units of $\alpha$)?
3. What does this imply for $\mathbb E_{a\sim\mu}[\hat Q(s,a)]$ compared with the true value?

**Step 1 — Zero, and the gradient.** If $\mu=\pi_\beta$ the two expectations coincide and the penalty vanishes for every $Q$. In general the penalty is $\alpha\sum_a\big(\mu(a\mid s)-\pi_\beta(a\mid s)\big)Q(s,a)$, linear in $Q$, with gradient $\alpha\big(\mu(a\mid s)-\pi_\beta(a\mid s)\big)$ with respect to $Q(s,a)$: gradient descent **lowers** $Q(s,a)$ where the learned policy puts more mass than the data ($\mu>\pi_\beta$) and **raises** it where it puts less. On its own the penalty is unbounded below; it is the TD term that keeps the values anchored, and the two together produce the shifted fixed point of the next step.

**Step 2 — Fixed point.** The correction $-\alpha(\mu/\pi_\beta-1)$ (which is the gradient of Step 1 divided by $\pi_\beta$, the weight with which each action appears in the TD term) gives, per action:

$\displaystyle a_1:\ \ -\alpha\Big(\frac{0.2}{0.8}-1\Big) = +0.75\alpha, \qquad a_2:\ \ -\alpha\Big(\frac{0.8}{0.2}-1\Big) = -3\alpha.$

The action the learned policy favours but the data rarely contain ($a_2$) is penalised by $3\alpha$; the well-supported action is actually *raised* by $0.75\alpha$.

**Step 3 — Lower bound.** Averaging over $\mu$: $\ \mathbb E_\mu[Q_\theta] = \mathbb E_\mu[\mathcal B^\mu Q_\theta] - \alpha\,\mathbb E_\mu\big[\tfrac{\mu}{\pi_\beta}-1\big]$, and $\mathbb E_\mu[\mu/\pi_\beta]-1 = \sum_a\mu^2/\pi_\beta - 1 \ge 0$ (here $0.05+3.2-1 = 2.25$). The learned value under the learned policy is **below** the Bellman backup by $2.25\alpha$: iterating, $\mathbb E_\mu[Q_\theta(s,\cdot)]$ is a lower bound of the true $v_\mu(s)$, with a gap that grows with how far $\mu$ strays from the data (while individual values, such as that of $a_1$ above, can be raised). Choosing $\alpha$ large enough guarantees the bound; choosing it too large makes $\mu$ cling to $\pi_\beta$ and CQL degenerates into imitation.

**Key concept**

CQL's penalty vanishes on the behaviour distribution and shifts the fixed point wherever the learned policy deviates from it, so the learned value of the learned policy under-estimates $v_\mu$ by an amount proportional to $\alpha$ and to the mismatch $\mu/\pi_\beta$. Conservatism is a lower bound on the value of exactly the policy being learned.

### Exercise 12.5 — Returns-to-go and the token sequence

An episode has rewards $(R_1,R_2,R_3,R_4)=(2,\,-1,\,3,\,5)$ (undiscounted, as in the Decision Transformer).

1. Compute the returns-to-go $\hat R_0,\dots,\hat R_3$.
2. Write the token sequence fed to the model for this episode, and state which tokens the output at the position of $S_2$ may attend to under the causal mask.
3. At test time the model is prompted with $\hat R_0=9$ and receives $R_1=2$, $R_2=-1$. What returns-to-go does it feed at steps $1$ and $2$? What if it had been prompted with $\hat R_0=20$?

**Step 1 — Returns-to-go** $\hat R_t=\sum_{t'\ge t}R_{t'+1}$, computed backwards: $\hat R_3=5$, $\hat R_2=3+5=8$, $\hat R_1=-1+8=7$, $\hat R_0=2+7=9$.

**Step 2 — Sequence and mask.** $\ \tau = (\hat R_0,S_0,A_0,\ \hat R_1,S_1,A_1,\ \hat R_2,S_2,A_2,\ \hat R_3,S_3,A_3) = (9,S_0,A_0,\ 7,S_1,A_1,\ 8,S_2,A_2,\ 5,S_3,A_3)$, twelve tokens. The token $S_2$ is the eighth; under the causal mask its output attends to tokens $1$–$8$: $\hat R_0,S_0,A_0,\hat R_1,S_1,A_1,\hat R_2,S_2$ — everything up to and including the current state and the current return-to-go, but **not** $A_2$, which is what it must predict, nor anything later.

**Step 3 — Prompting.** With $\hat R_0=9$: after $R_1=2$ the model is fed $\hat R_1=9-2=7$; after $R_2=-1$, $\hat R_2=7-(-1)=8$ — exactly the values seen in training for this trajectory, so the model is asked to continue a trajectory it recognises. With $\hat R_0=20$: $\hat R_1=18$, $\hat R_2=19$, values that never appeared in the data (the best episode returned $9$); the model must extrapolate and, having no trajectories to imitate, typically behaves like the best trajectories it knows — the achieved return saturates at the dataset's best.

**Key concept**

The return-to-go labels every state with "what followed", so the model learns $P(A_t\mid \hat R_t, S_t, \text{context})$. Decrementing the prompt by the rewards received keeps the model on the trajectories it was trained on; asking for more than the data contain is extrapolation, which a sequence model answers by imitating its best data rather than by inventing.

### Exercise 12.6 — One attention head by hand

Three tokens have (two-dimensional) queries, keys and values $q_1=(1,0),\ q_2=(0,1),\ q_3=(1,1)$; $k_1=(1,0),\ k_2=(0,1),\ k_3=(1,1)$; $v_1=(1,0),\ v_2=(0,1),\ v_3=(2,2)$; scaling $\sqrt d=\sqrt2$.

1. Compute the attention weights of token $3$ over tokens $1,2,3$ **without** a causal mask, and its output $z_3$.
2. Compute the weights and output of token $1$ **with** a causal mask.
3. Explain what the mask does to the information flow in a Decision Transformer.

**Step 1 — Token 3, no mask.** Scores $q_3^\top k_j/\sqrt2$: $(1\cdot1+1\cdot0)/\sqrt2 = 0.707$, $(0+1)/\sqrt2=0.707$, $(1+1)/\sqrt2 = 1.414$. Softmax: $e^{0.707}=2.028$, $2.028$, $e^{1.414}=4.113$; sum $8.169$; weights $(0.248,\,0.248,\,0.503)$.

$\displaystyle z_3 = 0.248(1,0)+0.248(0,1)+0.503(2,2) = (1.255,\ 1.255).$

Token $3$ attends mostly to itself (its key matches its query best) and equally to the other two.

**Step 2 — Token 1, causal.** Token $1$ may attend only to token $1$: the scores of tokens $2,3$ are set to $-\infty$, their weights are $0$, the weight of token $1$ is $1$, and $z_1=v_1=(1,0)$. Its output cannot depend on anything that comes later, whatever the content.

**Step 3 — In a Decision Transformer.** The mask guarantees that the prediction of $A_t$ (read at the $S_t$ position) uses only $\hat R_0,S_0,A_0,\dots,\hat R_t,S_t$: the model cannot peek at the action it is supposed to predict, nor at future states or returns, so what it learns is a *policy* that can be executed step by step at test time, with the same information it had in training. Without the mask the training loss would be trivially minimised by copying $A_t$ from the next token and the model would be useless for control.

**Key concept**

Attention mixes value vectors with weights given by query–key similarity; the causal mask zeroes the weights of future tokens, which is what makes the transformer a *sequential predictor* and the Decision Transformer a policy that can act online from its own history.

### Exercise 12.7 — Behaviour cloning vs. return conditioning on a toy dataset

A one-state dataset contains $100$ episodes with a single step. In $60$ of them the action was $L$ and the reward $0$; in $40$ the action was $R$ and the reward $10$.

1. What policy does behaviour cloning learn, and what is its greedy action?
2. What conditional distribution $P(A\mid\hat R)$ does a Decision Transformer learn, and what does it do when prompted with $\hat R=10$? With $\hat R=0$? With $\hat R=5$?
3. In what sense is the Decision Transformer "using the rewards" and behaviour cloning not?

**Step 1 — Cloning.** Maximum likelihood gives $\pi(L)=0.6$, $\pi(R)=0.4$; the greedy action is $L$, the *worse* one. Cloning reproduces the frequency of actions in the data, blind to their outcomes.

**Step 2 — Return conditioning.** The data contain $(\hat R=0,A=L)$ sixty times and $(\hat R=10,A=R)$ forty times, so $P(A=R\mid\hat R=10)=1$ and $P(A=L\mid\hat R=0)=1$. Prompted with $10$ the model takes $R$ and obtains $10$; prompted with $0$ it takes $L$ and obtains $0$; prompted with $5$, a value never seen, the behaviour depends on how the model generalises between the two embeddings — typically it interpolates, giving a mixture, which is one reason to prompt with returns that the data actually contain (or with the dataset maximum).

**Step 3 — Using the rewards.** Behaviour cloning never reads the reward column; its target is the action. The Decision Transformer reads the rewards, but only through the *conditioning variable*: it learns which actions went with which returns, and lets the user pick the return. It does not compute values or compare actions; it is still supervised learning, with the return as an extra input. This is why it is sometimes described as "return-conditioned behaviour cloning".

**Key concept**

With mixed-quality data, cloning imitates the majority; a return-conditioned model separates the data by outcome and can be steered to the good part. The improvement comes from conditioning, not from any form of value estimation.

### Exercise 12.8 — Stitching: what sequence models cannot do (integrative)

A dataset on a small deterministic MDP contains exactly three kinds of trajectory, all with $\gamma=1$: (i) $A\xrightarrow{a_1}B$ with reward $0$, then from $B$ a bad action with reward $0$ (return $0$); (ii) $A\xrightarrow{a_2}D$ with reward $5$, then the episode ends (return $5$); (iii) starting from $B$, the good action to $C$ with reward $10$ (return $10$). No trajectory goes $A\to B\to C$.

1. What is the best return achievable from $A$ by combining the data?
2. Can CQL (with the penalty small enough to allow the good action in $B$) learn it? Explain through the Bellman backup.
3. Which action does a Decision Transformer prompted with $\hat R_0=10$ in $A$ take? Explain through what it was trained on.

**Step 1 — Best return.** Take $a_1$ in $A$ (as in trajectories (i)), then the good action in $B$ (as in trajectories (iii)): return $0+10=10$, better than the $5$ of $a_2$. The dataset contains both pieces, but never together.

**Step 2 — CQL.** The Bellman backup propagates values across transitions regardless of which trajectory they came from: from (iii), $Q(B,\text{good})\approx10$; then the target for $(A,a_1)$ is $r+\gamma\max_{a'}Q(B,a') = 0+10 = 10$ (the good action in $B$ *is* in the data, so the conservative penalty leaves its value intact), while $Q(A,a_2)=5$. The greedy policy from $A$ takes $a_1$ and obtains $10$. Value functions **stitch**: they combine the best continuation of any state with any trajectory that reaches it.

**Step 3 — Decision Transformer.** In the training data, the windows starting in $A$ carry either $(\hat R_0=0, a_1)$ or $(\hat R_0=5, a_2)$; the pair $(\hat R=10,\ S=A)$ never occurs. Prompted with $10$ in $A$, the model is off its training distribution, and the closest return it has seen in $A$ is $5$, associated with $a_2$: it takes $a_2$ and obtains $5$. It has learned $P(A_t\mid\hat R=10,S=B)=\text{good}$, but it never gets to $B$. The model can only reproduce return–behaviour associations that appear in the data; it has no mechanism to infer that trajectories (i) and (iii) can be joined, because it never computes "the value of $B$".

**Key concept**

Stitching requires propagating value *across* trajectories, which is what bootstrapping does and sequence modelling does not. This is the fundamental trade-off of offline RL: value-based methods can exceed every trajectory in the data but must be made conservative; return-conditioned sequence models are stable and scalable but bounded by the trajectories they have seen.